# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Diagnóstico pre-integración

**Flujo de trabajo:**
1. Cargar archivos `.h5ad`
2. Figuras y CSV + objeto actualizado con UMAP y vecindario pre-integración.

Este paso actúa como "control negativo", de esta forma, si tras Harmony las células se mezclan por condición biológica en lugar de por muestra, la corrección habrá funcionado.

# · Importaciones y configuración

In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda.
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:12
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend sin pantalla — guarda figuras sin mostrarlas
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 2

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

INPUT_DIR   = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['02_normalized']}"
OUTPUT_DIR  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['03_diagnostics']}"
INPUT_PATH  = f"{INPUT_DIR}/{PARAMS['outputs']['normalized_h5ad']}"
OUTPUT_PATH = f"{OUTPUT_DIR}/{PARAMS['outputs']['diagnosed_h5ad']}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables'].get('03_diagnostics', 'reports/tables/03_diagnostics')}"
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures']['03_diagnostics']}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR


BATCH_KEY     = "sample_id"
CONDITION_KEY = "condition"   # se añade automáticamente abajo

N_PCS_HARMONY = 20   # según el elbow plot anterior

In [ ]:
adata = sc.read_h5ad(INPUT_PATH)
print(f"   {adata.n_obs:,} células × {adata.n_vars:,} genes")
print(f"   obsm disponibles: {list(adata.obsm.keys())}")

# Verificar que el PCA existe antes de continuar
# (puede faltar si el Paso 2 no se completó correctamente)
assert 'X_pca' in adata.obsm, (
    "ERROR: 'X_pca' no encontrado en adata.obsm. "
    "Ejecuta primero el Paso 2 (normalización)."
)
n_pcs_available = adata.obsm['X_pca'].shape[1]
assert N_PCS_HARMONY <= n_pcs_available, (
    f"ERROR: N_PCS_HARMONY={N_PCS_HARMONY} supera las "
    f"{n_pcs_available} PCs disponibles en X_pca. "
    f"Reduce N_PCS_HARMONY o recalcula PCA con más componentes en el Paso 2."
)
print(f"   X_pca shape: {adata.obsm['X_pca'].shape}")
print(f"   Usando primeras {N_PCS_HARMONY} de {n_pcs_available} PCs")


   43,388 células × 33,538 genes
   obsm disponibles: ['X_pca']
   X_pca shape: (43388, 50)
   Usando primeras 20 de 50 PCs


# · Añadir metadato de condición (HC / UC / CD)

Los sample_ids siguen el patrón GSMxx_HC-1, GSMxx_UC-3, etc.
Se extrae la condición del nombre para poder colorear los gráficos por grupo biológico además de por muestra.

In [ ]:
def extract_condition(sample_id):
    # Extrae HC, UC o CD del nombre de la muestra

    for cond in ['HC', 'UC', 'CD']:
        if cond in sample_id:
            return cond
    return 'Unknown'

adata.obs[CONDITION_KEY] = adata.obs[BATCH_KEY].apply(extract_condition)
print(f"\n→ Condiciones detectadas:")
print(adata.obs[CONDITION_KEY].value_counts().to_string())

# Paleta de colores por condición
PALETTE = {'HC': '#2E86AB', 'UC': '#E84855', 'CD': '#F4A261'}
CONDITIONS = ['HC', 'UC', 'CD']



→ Condiciones detectadas:
condition
CD    15076
UC    14210
HC    14102


# · Estadísticas descriptivas por muestra


In [ ]:
# Recalcular métricas QC si no están en .obs
if 'n_genes_by_counts' not in adata.obs.columns:
    print("   n_genes_by_counts no encontrado — recalcular métricas QC")
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
    )

stats = adata.obs.groupby(BATCH_KEY).agg(
    n_cells      = (BATCH_KEY, 'count'),
    med_genes    = ('n_genes_by_counts', 'median'),
    med_umis     = ('total_counts', 'median'),
    med_mito_pct = ('pct_counts_mt', 'median'),
).round(1)
stats['condition'] = stats.index.map(extract_condition)

print(stats.to_string())
stats_path = f"{TABLES_DIR}/pre_integration_stats_per_sample.csv"
stats.to_csv(stats_path)
print(f"→ CSV guardado: {stats_path}")

# Advertencia para muestras con mito elevado
high_mito = stats[stats['med_mito_pct'] > 30]
if len(high_mito) > 0:
    print(f"Muestras con mediana % mito > 30%:")
    print(high_mito[['condition', 'n_cells', 'med_mito_pct']].to_string())


                 n_cells  med_genes  med_umis  med_mito_pct condition
sample_id                                                            
GSM6614348_HC-1     1455     1000.0    3182.0      3.800000        HC
GSM6614349_HC-2     2848     1429.0    4784.0      9.000000        HC
GSM6614350_HC-3     3082      990.0    3143.5     15.000000        HC
GSM6614351_HC-4     2031      966.0    3024.0     18.900000        HC
GSM6614352_HC-5     3082     1104.0    3562.0     13.400000        HC
GSM6614353_HC-6     1604     1071.0    3598.0     16.200001        HC
GSM6614354_UC-1     1311      857.0    2949.0      1.800000        UC
GSM6614355_UC-2     2451     1119.0    7143.0      4.500000        UC
GSM6614356_UC-3     2033     1096.0    5318.0      6.800000        UC
GSM6614357_UC-4     2922     1280.5    4278.0      7.900000        UC
GSM6614358_UC-5     2583     1590.0    9897.0      3.600000        UC
GSM6614359_UC-6     2910     1068.0    3890.0      8.800000        UC
GSM6614360_CD-1     

# · Figura 1: Violin plots por condición


In [ ]:
print("\n→ Figura 1: violin plots de métricas QC por condición")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Métricas QC por condición biológica - pre-integración",
             fontsize=11, fontweight='bold')

for ax, metric, label in zip(
    axes,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    ['Genes / célula', 'UMIs / célula', '% Mitocondrial']
):
    # Usar palette[cond] directamente
    data_by_cond = [
        adata.obs[metric][adata.obs[CONDITION_KEY] == cond].values
        for cond in CONDITIONS
    ]
    parts = ax.violinplot(data_by_cond, showmedians=True, widths=0.7)
    for pc, cond in zip(parts['bodies'], CONDITIONS):
        pc.set_facecolor(PALETTE[cond])
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('black')
    ax.set_xticks(range(1, len(CONDITIONS) + 1))
    ax.set_xticklabels(CONDITIONS)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/pre_integration_QC_violins.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("→ Guardada: pre_integration_QC_violins.png")



→ Figura 1: violin plots de métricas QC por condición...
   → Guardada: pre_integration_QC_violins.png


# · Figura 2: UMAP batch vs biología





In [ ]:
# Calculamos el grafo KNN sobre el PCA sin corregir y proyectamos en UMAP.
# key_added='neighbors_uncorrected' guarda el vecindario con un nombre
# específico para no sobrescribir el que calcularemos tras Harmony.

sc.pp.neighbors(
    adata,
    n_pcs=N_PCS_HARMONY,
    use_rep='X_pca',
    key_added='neighbors_uncorrected'
)

print("\n→ Calculando UMAP sin corrección de batch...")
sc.tl.umap(
    adata,
    neighbors_key='neighbors_uncorrected',
    random_state=42
)
# Guardar con nombre específico para no perderlo al calcular el UMAP corregido
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()
print("Guardado en obsm['X_umap_uncorrected']")


computing neighbors
    finished (0:02:25)

→ Calculando UMAP sin corrección de batch...
computing UMAP
    finished (0:01:09)
Guardado en obsm['X_umap_uncorrected']


In [ ]:
print("\n→ Figura 2: UMAP batch effect vs señal biológica...")
umap  = adata.obsm['X_umap_uncorrected']
samples    = adata.obs[BATCH_KEY].values
conditions = adata.obs[CONDITION_KEY].values
sample_list = sorted(adata.obs[BATCH_KEY].unique())

# Paleta de colores para las 18 muestras
cmap_samples = plt.get_cmap('tab20', len(sample_list))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("UMAP sin corrección de batch — pre-integración",
             fontsize=12, fontweight='bold')

# Panel izquierdo: por muestra
ax = axes[0]
for i, s in enumerate(sample_list):
    mask = samples == s
    ax.scatter(umap[mask, 0], umap[mask, 1],
               s=1, alpha=0.3, color=cmap_samples(i), label=s, rasterized=True)
ax.set_title("Por muestra (sample_id)",
             fontsize=10)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc='upper left',
          fontsize=6, frameon=False)

# Panel derecho: por condición
ax = axes[1]
for cond in CONDITIONS:
    mask = conditions == cond
    ax.scatter(umap[mask, 0], umap[mask, 1],
               s=1, alpha=0.3, color=PALETTE[cond], label=cond, rasterized=True)
ax.set_title("Por condición (HC/UC/CD)",
             fontsize=10)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.legend(markerscale=8, fontsize=10, frameon=False)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/UMAP_pre_integration_batch_vs_biology.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("→ Guardada: UMAP_pre_integration_batch_vs_biology.png")



→ Figura 2: UMAP batch effect vs señal biológica...
   → Guardada: UMAP_pre_integration_batch_vs_biology.png


# · Figura 3: UMAP métricas técnicas


In [ ]:
print("\n→ Figura 3: UMAP métricas técnicas...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("UMAP sin corrección — métricas técnicas\n",
             fontsize=11, fontweight='bold')

for ax, metric, label, cmap in zip(
    axes,
    ['pct_counts_mt', 'n_genes_by_counts'],
    ['% Mitocondrial', 'Nº genes / célula'],
    ['RdYlGn_r', 'viridis']
):
    vals = adata.obs[metric].values
    sc_ = ax.scatter(umap[:, 0], umap[:, 1],
                     c=vals, cmap=cmap,
                     s=0.5, alpha=0.3, rasterized=True)
    plt.colorbar(sc_, ax=ax, label=label, shrink=0.8)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/UMAP_pre_integration_technical_metrics.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("→ Guardada: UMAP_pre_integration_technical_metrics.png")



→ Figura 3: UMAP métricas técnicas...
   → Guardada: UMAP_pre_integration_technical_metrics.png


# · Figura 4: PCA coloreado por muestra


In [ ]:
print("\n→ Figura 4: PCA por muestra (primeras 4 componentes)...")
pca = adata.obsm['X_pca']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("PCA sin corrección de batch — primeras 4 componentes\n",
             fontsize=11, fontweight='bold')

for ax, (pcx, pcy) in zip(axes.flatten(), [(0,1), (0,2), (1,2), (2,3)]):
    for i, s in enumerate(sample_list):
        mask = samples == s
        ax.scatter(pca[mask, pcx], pca[mask, pcy],
                   s=0.5, alpha=0.3, color=cmap_samples(i), rasterized=True)
    ax.set_xlabel(f"PC{pcx+1}"); ax.set_ylabel(f"PC{pcy+1}")
    ax.set_title(f"PC{pcx+1} vs PC{pcy+1}", fontsize=9)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/PCA_pre_integration_by_sample.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("→ Guardada: PCA_pre_integration_by_sample.png")



→ Figura 4: PCA por muestra (primeras 4 componentes)...
   → Guardada: PCA_pre_integration_by_sample.png


# · GUARDAR



In [ ]:
print(f"\n→ Guardando objeto en {OUTPUT_PATH} ...")
adata.write_h5ad(OUTPUT_PATH, compression='gzip')


→ Guardando objeto en /content/drive/MyDrive/IBD_TFM/data/interim/03_diagnostics/IBD_diagnosed.h5ad ...


# · Análisis específico elevado % mitocondrial

Las muestras HC-3 (15%), HC-4 (18.9%), HC-5 (13.4%) y HC-6 (16.2%) tienen medianas de % mito elevadas para ser controles sanos.



In [ ]:
HC_HIGH_MITO = ['GSM6614350_HC-3', 'GSM6614351_HC-4',
                'GSM6614352_HC-5', 'GSM6614353_HC-6']

In [ ]:
# ── Figura A: UMAP mito — todas las muestras vs sin HC problemáticas ──────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Izquierda: todas las células | Derecha: sin HC-3/4/5/6",
    fontsize=11, fontweight='bold'
)

umap_all = adata.obsm['X_umap_uncorrected']
mito_all  = adata.obs['pct_counts_mt'].values
samples_all = adata.obs[BATCH_KEY].values

# Panel izquierdo: todas las células
ax = axes[0]
sc_ = ax.scatter(
    umap_all[:, 0], umap_all[:, 1],
    c=mito_all, cmap='RdYlGn_r',
    vmin=0, vmax=50,
    s=0.5, alpha=0.4, rasterized=True
)
plt.colorbar(sc_, ax=ax, label='% Mitocondrial', shrink=0.8)
ax.set_title("Todas las células\n(incluyendo HC-3/4/5/6)", fontsize=10)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

# Panel derecho: excluir HC problemáticas
mask_no_hc_prob = ~np.isin(samples_all, HC_HIGH_MITO)
umap_sub = umap_all[mask_no_hc_prob]
mito_sub  = mito_all[mask_no_hc_prob]
cond_sub  = adata.obs[CONDITION_KEY].values[mask_no_hc_prob]
n_removed = (~mask_no_hc_prob).sum()

ax = axes[1]
sc_ = ax.scatter(
    umap_sub[:, 0], umap_sub[:, 1],
    c=mito_sub, cmap='RdYlGn_r',
    vmin=0, vmax=50,
    s=0.5, alpha=0.4, rasterized=True
)
plt.colorbar(sc_, ax=ax, label='% Mitocondrial', shrink=0.8)
ax.set_title(
    f"Sin HC-3/4/5/6 ({n_removed:,} células excluidas)\n",
    fontsize=9
)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/mito_diagnostic_with_vs_without_HC_problematic.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"→ Figura guardada: mito_diagnostic_with_vs_without_HC_problematic.png")



  → Figura guardada: mito_diagnostic_with_vs_without_HC_problematic.png


In [ ]:
# ── Figura B: distribución de mito por condición, separando HC prob vs resto ─
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Distribución de % mitocondrial por grupo\n",
    fontsize=11, fontweight='bold'
)

# Grupos a comparar
groups = {
    'HC sanos\n(HC-1, HC-2)':
        adata.obs['pct_counts_mt'][
            adata.obs[BATCH_KEY].isin(['GSM6614348_HC-1', 'GSM6614349_HC-2'])
        ].values,
    'HC problem.\n(HC-3/4/5/6)':
        adata.obs['pct_counts_mt'][
            adata.obs[BATCH_KEY].isin(HC_HIGH_MITO)
        ].values,
    'UC + CD\n(todos)':
        adata.obs['pct_counts_mt'][
            adata.obs[CONDITION_KEY].isin(['UC', 'CD'])
        ].values,
}
colors_g = ['#2E86AB', '#E84855', '#F4A261']

for ax, (label, vals), color in zip(axes, groups.items(), colors_g):
    ax.hist(vals, bins=60, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(np.median(vals), color='black', lw=2, ls='-',
               label=f'Mediana: {np.median(vals):.1f}%')
    ax.axvline(15, color='red', lw=1.5, ls='--', label='15% (límite colon sano)')
    ax.set_xlabel('% Mitocondrial', fontsize=9)
    ax.set_ylabel('Nº células', fontsize=9)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/mito_diagnostic_distribution_by_group.png",
            dpi=120, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"→ Figura guardada: mito_diagnostic_distribution_by_group.png")


  → Figura guardada: mito_diagnostic_distribution_by_group.png
